In [6]:
import pandas as pd


unicos = pd.read_csv('../Data/Processed/Dataset_Unicos.csv')
exploded = pd.read_csv('../Data/Processed/Dataset_Expandido.csv')

Harman_in_ear = pd.read_csv('../Data/Raw/Harman in-ear 2019.csv')
Harman_over_ear = pd.read_csv('../Data/Raw/Harman over-ear 2018.csv')

# Primero asegúrate que todas las frecuencias son float y redondeadas a 1 decimal
exploded["Frecuencia"] = exploded["Frecuencia"].astype(float).round(1)
Harman_over_ear["frequency"] = Harman_over_ear["frequency"].astype(float).round(1)
Harman_in_ear["frequency"] = Harman_in_ear["frequency"].astype(float).round(1)

Harman_over_ear = Harman_over_ear.rename(columns={"frequency": "Frecuencia", "level": "Respuesta_Harman"})
Harman_in_ear = Harman_in_ear.rename(columns={"frequency": "Frecuencia", "level": "Respuesta_Harman"})


In [7]:
exploded = exploded.merge(
    unicos[["MarcaReferencia", "TipoAudifono"]],
    on="MarcaReferencia",
    how="left"
)

In [8]:
# Dividir por tipo
df_over = exploded[exploded["TipoAudifono"].str.lower() == "over-ear"]
df_in = exploded[exploded["TipoAudifono"].str.lower().isin(["in-ear", "earbuds"])]

# Hacer merge por frecuencia
df_over = df_over.merge(Harman_over_ear, on="Frecuencia", how="left", suffixes=("", "_Harman"))
df_in = df_in.merge(Harman_in_ear, on="Frecuencia", how="left", suffixes=("", "_Harman"))

# Unir nuevamente
df_comparado = pd.concat([df_over, df_in], ignore_index=True)


In [10]:
# Error absoluto
df_comparado["ErrorAbs"] = (df_comparado["Respuesta"] - df_comparado["raw"]).abs()

# Error cuadrático (para luego sacar RMSE por audífono)
df_comparado["ErrorCuad"] = (df_comparado["Respuesta"] - df_comparado["raw"]) ** 2



In [11]:
mae = df_comparado.groupby("MarcaReferencia")["ErrorAbs"].mean().reset_index(name="MAE")

# RMSE por audífono
rmse = df_comparado.groupby("MarcaReferencia")["ErrorCuad"].mean().pow(0.5).reset_index(name="RMSE")

# Unir
df_metricas = mae.merge(rmse, on="MarcaReferencia")

In [13]:
df_metricas["Affinity"] = 100 - df_metricas["RMSE"] * 10
df_metricas["Affinity"] = df_metricas["Affinity"].clip(lower=0)

In [15]:
from sklearn.preprocessing import KBinsDiscretizer

# Supón que tienes un DataFrame llamado df_metricas con la columna 'Affinity'
kbins = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='kmeans')
df_metricas["ClusterAffinity"] = kbins.fit_transform(df_metricas[["Affinity"]]).astype(int)

In [22]:
mapa_etiquetas = {
    3: "Estilo Harman – Usuario general",
    2: "Buen tono – Musical equilibrado",
    1: "Personalizado – Firma no neutra",
    0: "Alternativo"
}

df_metricas["Segmento"] = df_metricas["ClusterAffinity"].map(mapa_etiquetas)

In [25]:
df_metricas

,MarcaReferencia,MAE,RMSE,Affinity,ClusterAffinity,Segmento
0,AKG K1000,4.838950,8.194478,18.055216,0,Alternativo
1,AKG K167 Tiesto,3.991856,4.668756,53.312438,2,Buen tono – Musical equilibrado
2,AKG K240 MKII,3.748906,5.286193,47.138071,2,Buen tono – Musical equilibrado
3,AKG K240 Monitor,2.868158,4.141181,58.588194,2,Buen tono – Musical equilibrado
4,AKG K240 Sextett,3.629209,4.423107,55.768929,2,Buen tono – Musical equilibrado
...,...,...,...,...,...,...
1147,SteelSeries Arctis Pro Wireless,2.256374,3.496590,65.034101,3,Estilo Harman – Usuario general
1148,SteelSeries Flux In-Ear,2.844144,3.710045,62.899546,3,Estilo Harman – Usuario general
1149,SteelSeries Flux InEar Pro,4.018504,6.275619,37.243815,1,Personalizado – Firma no neutra
1150,SteelSeries Siberia 200,2.810374,3.425221,65.747788,3,Estilo Harman – Usuario general


In [27]:
unicos = unicos.merge(
    df_metricas[["MarcaReferencia","MAE", "RMSE", "Affinity", "Segmento"]],
    on="MarcaReferencia",
    how="left"
)


In [29]:
unicos.to_csv('../Data/Processed/Dataset_Unicos_Affinity.csv', index=False)